# Mapper, Reducers customis?s avec SQL

Ce notebook propose l'utilisation de SQL avec [SQLite](https://sqlite.org/) pour manipuler les donn?es depuis un notebook (avec le module [sqlite3](https://docs.python.org/3.6/library/sqlite3.html)).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image

## Repr?sentation

Le module [pandas](https://pandas.pydata.org/) manipule des tables et c'est la fa?on la plus commune de repr?senter les donn?es. Lorsque les donn?es sont multidimensionnelles, on distingue les coordonn?es des valeurs :

In [ ]:
Image("cube1.png")

Dans cet exemple, il y a :
    
* 3 coordonn?es : Age, Profession, Ann?ee
* 2 valeurs : Esp?rance de vie, Population

On peut repr?senter les donn?s ?galement comme ceci :

In [ ]:
Image("cube2.png")

C'est assez simple. Prenons un exemple : [Tables de mortalit? par g?n?ration en Frances](https://www.insee.fr/fr/statistiques/6543678?sommaire=6543680) qu'on r?cup?re ? l'aide de la fonction [mortality_table](https://sdpython.github.io/doc/teachcompute/dev/index.html). C'est assez long (4-5 minutes) sur l'ensemble des donn?es car elles doivent ?tre pr?trait?es (voir la documentation de la fonction). Pour ?couter, il faut utiliser le param?tre *stop_at*.

In [ ]:
from teachcompute.datasets import mortality_table

filename = mortality_table(verbose=True)
filename

In [ ]:
import os

size = os.stat(filename).st_size
print(f"size={size / 2**20:1.3f} Mb")

In [ ]:
import pandas

df = pandas.read_csv(filename, sep="\t", encoding="utf-8", low_memory=False)
df.head()

Les indicateurs pour deux ?ges diff?rents :

In [ ]:
df[
    ((df.age == "Y60") | (df.age == "Y61"))
    & (df.annee == 2000)
    & (df.pays == "FR")
    & (df.genre == "F")
]

## Donn?es trop grosses pour tenir en m?moire : SQLite

On charge une grosse base de donn?es (assez petite pour que la s?ance ne soit pas trop longue).

In [ ]:
df.shape

Les donn?es sont trop grosses pour tenir dans une feuille Excel et les consulter il n'y a pas d'autres moyens que d'en regarder des extraits. Que passe-t-il quand les donn?es sont encore plus grosses et qu'elles ne tiennent pas en m?moire ? Quelques solutions :

* augmenter la m?moire de l'ordinateur, avec 20 Go, on peut faire beaucoup de choses,
* stocker les donn?es dans un serveur SQL,
* stocker les donn?es sur un syst?me distribu? (cloud, Hadoop, ...)

La seconde option n'est pas toujours simple, il faut installer un serveur SQL. Pour aller plus vite, on peut simplement utiliser [SQLite](https://www.sqlite.org/) qui est une fa?on de faire du SQL sans serveur (cela prend quelques minutes). On utilise la m?thode [to_sql](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.to_sql.html).

In [ ]:
import sqlite3
from pandas.io import sql

cnx = sqlite3.connect("mortalite.db3")
try:
    df.to_sql(name="mortalite", con=cnx)
except ValueError as e:
    if "Table 'mortalite' already exists" not in str(e):
        # seulement si l'erreur ne vient pas du fait que cela
        # a d?j? ?t? fait
        raise e
# on peut ajouter d'autres dataframe ? la table comme si elle ?tait cr??e par morceau
# voir le param?tre if_exists de la fonction to_sql

On peut maintenant r?cup?rer un morceau avec la fonction [read_sql](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.read_sql.html?highlight=read_sql#pandas.read_sql).

In [ ]:
import pandas

example = pandas.read_sql("SELECT * FROM mortalite WHERE age_num==50 LIMIT 5", cnx)
example

L'ensemble des donn?es restent sur le disque, seul le r?sultat de la requ?te est charg? en m?moire. Si on ne peut pas faire tenir les donn?es en m?moire, il faut soit en obtenir une vue partielle (un ?chantillon al?atoire, un vue filtr?e), soit une vue agr?gr?e. Pour finir, il faut fermer la connexion pour laisser d'autres applications ou notebook modifier la base ou tout simplement supprimer le fichier.

In [ ]:
cnx.close()

Sous Windows, on peut consulter la base avec le logiciel [SQLiteSpy](https://www.yunqa.de/delphi/apps/sqlitespy/index).

In [ ]:
Image("sqlite.png")

Sous Linux ou Max, on peut utiliser une extension Firefox [SQLite Manager](https://addons.mozilla.org/en-US/firefox/addon/sqlite-manager-webext/).

## Cas 1 : filtrer pour cr?er un ?chantillon al?atoire

Si on ne peut pas faire tenir les donn?es en m?moire, on peut soit regarder les premi?res lignes soit prendre un ?chantillon al?atoire. Deux options :

* [Dataframe.sample](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.sample.html)
* [create_function](https://docs.python.org/3.4/library/sqlite3.html#sqlite3.Connection.create_function)

La premi?re fonction est simple :

In [ ]:
sample = df.sample(frac=0.1)
sample.shape, df.shape

Je ne sais pas si cela peut ?tre r?alis? sans charger les donn?es en m?moire. Si les donn?es p?sent 20 Go, cette m?thode n'aboutira pas. Pourtant, on veut juste un ?chantillon pour commencer ? regarder les donn?es. On utilise la seconde option avec [create_function](https://docs.python.org/3.4/library/sqlite3.html#sqlite3.Connection.create_function) et la fonction suivante :

In [ ]:
import random  # loi uniforme


def echantillon(proportion):
    return 1 if random.random() < proportion else 0

In [ ]:
import sqlite3
from pandas.io import sql

cnx = sqlite3.connect("mortalite.db3")

On d?clare la fonction ? la base de donn?es.

In [ ]:
cnx.create_function("echantillon", 1, echantillon)

On veut r?cup?rer environ 1% de la table ? On ?crit d'abord le filtre.

In [ ]:
sample = pandas.read_sql("SELECT * FROM mortalite WHERE echantillon(0.01)", cnx)
sample.shape

In [ ]:
sample.head()

On ferme la connexion.

In [ ]:
cnx.close()

## Pseudo Map/Reduce avec SQLite

La liste des [mots-cl?s du langage SQL utilis?s par SQLite](https://www.sqlite.org/keyword_index.html) n'est pas aussi riche que d'autres solutions de serveurs SQL. La m?diane ne semble pas en faire partie. Cependant, pour une ann?e, un genre, un ?ge donn?, on voudrait calculer la m?diane de l'esp?rance de vie sur l'ensembles des pays.

In [ ]:
import sqlite3, pandas
from pandas.io import sql

cnx = sqlite3.connect("mortalite.db3")

In [ ]:
pays = pandas.read_sql("SELECT pays, COUNT(*) FROM mortalite GROUP BY pays", cnx)
pays.head()

Il n'y a pas le m?me nombre de donn?es selon les pays, il est probable que le nombre de pays pour lesquels il existe des donn?es varie selon les ?ges et les ann?es.

In [ ]:
query = """SELECT nb_country, COUNT(*) AS nb_rows FROM (
                SELECT annee,age,age_num, count(*) AS nb_country FROM mortalite 
                WHERE indicateur=="LIFEXP" AND genre=="F"
                GROUP BY annee,age,age_num
            ) GROUP BY nb_country"""
df = pandas.read_sql(query, cnx)

In [ ]:
df.sort_values("nb_country", ascending=False).head(n=2)

In [ ]:
ax = df.plot(x="nb_country", y="nb_rows")
ax.set_title("Nombre de donn?es par pays");

Soit un nombre inconstant de pays. Le fait qu'on est 100 pays sugg?re qu'on ait une erreur ?galement.

In [ ]:
query = """SELECT annee,age,age_num, count(*) AS nb_country FROM mortalite 
                WHERE indicateur=="LIFEXP" AND genre=="F"
                GROUP BY annee,age,age_num
                HAVING nb_country >= 100"""
df = pandas.read_sql(query, cnx)

In [ ]:
df.head()

Ce sont des valeurs manquantes. Le probl?me pour calculer la m?diane pour chaque observation est qu'il faut d'abord regrouper les lignes de la table par indicateur puis choisir la m?diane dans chaque de ces petits groupes. On s'inspire pour cela de la logique Map/Reduce et de la fonction [create_aggregate](https://docs.python.org/3.4/library/sqlite3.html#sqlite3.Connection.create_aggregate).

## Cas 2 : reducer customis? avec SQL

Le reducer se pr?sente toujours sous la forme suivante :

In [ ]:
class ReducerMediane:
    def __init__(self):
        # ???
        pass

    def step(self, value):
        # ???
        #
        pass

    def finalize(self):
        # ???
        # return ... //2 ]
        pass

Qu'on renseigne de la sorte :

In [ ]:
class ReducerMediane:
    def __init__(self):
        self.indicateur = []

    def step(self, value):
        if value >= 0:
            self.indicateur.append(value)

    def finalize(self):
        self.indicateur.sort()
        return self.indicateur[len(self.indicateur) // 2]

On le d?clare ensuite ? *sqllite3*.

In [ ]:
cnx.create_aggregate("ReducerMediane", 1, ReducerMediane)

In [ ]:
query = """SELECT annee,age,age_num, ReducerMediane(valeur) AS mediane FROM mortalite 
                WHERE indicateur=="LIFEXP" AND genre=="F"
                GROUP BY annee,age,age_num"""
df = pandas.read_sql(query, cnx)

In [ ]:
df.head()

Un reducer ? deux entr?es m?me si cela n'a pas beaucoup de sens ici :

In [ ]:
class ReducerMediane2:
    def __init__(self):
        self.indicateur = []

    def step(self, value, value2):
        if value >= 0:
            self.indicateur.append(value)
        if value2 >= 0:
            self.indicateur.append(value2)

    def finalize(self):
        self.indicateur.sort()
        return self.indicateur[len(self.indicateur) // 2]


cnx.create_aggregate("ReducerMediane2", 2, ReducerMediane2)

In [ ]:
query = """SELECT annee,age,age_num, ReducerMediane2(valeur, valeur+1) AS mediane2 FROM mortalite 
                WHERE indicateur=="LIFEXP" AND genre=="F"
                GROUP BY annee,age,age_num"""
df = pandas.read_sql(query, cnx)
df.head()

Il n'est apparemment pas possible de retourner deux r?sultats mais on peut utiliser une ruse qui consise ? les concat?ner dans une cha?ne de carac?res.

In [ ]:
class ReducerQuantile:
    def __init__(self):
        self.indicateur = []

    def step(self, value):
        if value >= 0:
            self.indicateur.append(value)

    def finalize(self):
        self.indicateur.sort()
        q1 = self.indicateur[len(self.indicateur) // 4]
        q2 = self.indicateur[3 * len(self.indicateur) // 4]
        n = len(self.indicateur)
        return "%f;%f;%s" % (q1, q2, n)


cnx.create_aggregate("ReducerQuantile", 1, ReducerQuantile)

In [ ]:
query = """SELECT annee,age,age_num, ReducerQuantile(valeur) AS quantiles FROM mortalite 
                WHERE indicateur=="LIFEXP" AND genre=="F"
                GROUP BY annee,age,age_num"""
df = pandas.read_sql(query, cnx)
df.head()

On ferme la connexion.

In [ ]:
cnx.close()

## Notion d'index

En SQL et pour de grandes tables, la notion d'index joue un r?le important pour acc?l?rer les op?rations de jointures (``JOIN``) ou de regroupement (``GROUP BY``). L'article [A thorough guide to SQLite database operations in Python](http://sebastianraschka.com/Articles/2014_sqlite_in_python_tutorial.html) montre comment faire les principales op?rations.